# Learning RAG System - Summary Feature


## 1. Environment Setup

In [1]:
!pip uninstall -y torchcodec
!pip install -q langchain langchain-community langchain-core langchain-google-genai langchain-huggingface langchain-openai langchain-qdrant langchain-text-splitters loguru pypdf qdrant-client ragas sentence-transformers transformers typer vllm

!mkdir -p data
!gdown 1PqUYjfhdR8xBqj4mB7bW5X-yreYWqlZY -O data/data.zip
!unzip -o data/data.zip -d data

Found existing installation: torchcodec 0.10.0+cu128
Uninstalling torchcodec-0.10.0+cu128:
  Successfully uninstalled torchcodec-0.10.0+cu128
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 76.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 543.9/543.9 kB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.9/389.9 kB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.4/244.4 MB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 82.5 MB/

## 2. Core Imports

In [2]:
from pathlib import Path
from loguru import logger
import json
import re

from pydantic import BaseModel, Field
from typing import Literal, List, Optional
from pydantic_settings import BaseSettings

## 3. Data Schemas

In [3]:
class ChunkMetadata(BaseModel):
    document_id: str
    filename: str
    source: str
    page: int
    chunk_id: str
    section: Optional[str] = None

class RetrievedChunk(BaseModel):
    text: str
    score: float
    metadata: ChunkMetadata

class Citation(BaseModel):
    source_index: int
    source_marker: str
    filename: str
    page: int
    section: Optional[str] = None
    chunk_id: str

class Summary(BaseModel):
    scope: str
    target: Optional[str] = None
    summary: str
    key_points: List[str]
    citations: List[Citation]

## 4. System Configuration

In [4]:
class Settings(BaseSettings):
    data_dir: Path = Path("data")
    storage_dir: Path = Path("storage/qdrant")
    qdrant_collection: str = "rag_chunks"

    chunk_size: int = Field(default=1000, ge=100)
    chunk_overlap: int = Field(default=150, ge=0)

    embedding_model: str = "keepitreal/vietnamese-sbert"

    llm_provider: Literal["hf_local", "gemini", "vllm"] = "hf_local"
    llm_temperature: float = Field(default=0.1, ge=0.0, le=2.0)

    hf_model: str = "Qwen/Qwen2.5-1.5B-Instruct"
    hf_device: int = 0
    hf_max_new_tokens: int = Field(default=1024, ge=1)

    gemini_model: str = "gemini-2.5-flash"

    summarize_batch_size: int = Field(default=10, ge=1)
    summarize_retrieval_k: int = Field(default=12, ge=1, le=128)

settings = Settings()
settings.data_dir.mkdir(parents=True, exist_ok=True)
settings.storage_dir.mkdir(parents=True, exist_ok=True)

## 5. RAG Dependencies

In [5]:
import hashlib
import uuid
from collections import defaultdict

from langchain_community.document_loaders.pdf import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http import models as qmodels

## 6. Embedding Model

In [6]:
def get_embeddings():
    return HuggingFaceEmbeddings(
        model_name=settings.embedding_model,
        model_kwargs={"device": "cuda"},
        encode_kwargs={"normalize_embeddings": True},
    )

## 7. Vector Store Setup

In [7]:
def get_client() -> QdrantClient:
    return QdrantClient(path=str(settings.storage_dir))

def ensure_collection(recreate: bool = False):
    client = get_client()
    if recreate and client.collection_exists(settings.qdrant_collection):
        client.delete_collection(settings.qdrant_collection)

    if not client.collection_exists(settings.qdrant_collection):
        dim = len(get_embeddings().embed_query("test"))
        client.create_collection(
            collection_name=settings.qdrant_collection,
            vectors_config=qmodels.VectorParams(
                size=dim, distance=qmodels.Distance.COSINE),
        )

def get_vector_store():
    return QdrantVectorStore(
        client=get_client(),
        collection_name=settings.qdrant_collection,
        embedding=get_embeddings(),
    )

## 8. PDF Chunk Builder

In [8]:
def build_chunks(pdf_paths):
    page_docs = []

    for path in pdf_paths:
        loader = PyPDFLoader(str(path))
        pages = loader.load()

        doc_id = hashlib.sha1(f"{path.name}:{path.stat().st_size}".encode("utf-8")).hexdigest()[:16]

        for doc in pages:
            doc.metadata = {
                "document_id": doc_id,
                "filename": path.name,
                "source": str(path.resolve()),
                "page": int(doc.metadata.get("page", 0)) + 1,
            }
        page_docs.extend(pages)

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=settings.chunk_size,
        chunk_overlap=settings.chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    chunks = splitter.split_documents(page_docs)

    per_doc_counter = defaultdict(int)
    for chunk in chunks:
        doc_id = chunk.metadata["document_id"]
        idx = per_doc_counter[doc_id]
        per_doc_counter[doc_id] += 1
        chunk.metadata["chunk_id"] = f"{doc_id}:{chunk.metadata['page']}:{idx}"

    return chunks

## 9. Single PDF Ingestion

In [9]:
def save_and_ingest_pdf(file_path: str):
    path = Path(file_path)
    ensure_collection(recreate=False)
    chunks = build_chunks([path])
    if not chunks:
        return 0

    ids = [str(uuid.uuid5(uuid.NAMESPACE_DNS, c.metadata["chunk_id"]))
        for c in chunks]
    get_vector_store().add_documents(chunks, ids=ids)
    logger.info(f"Saved {len(chunks)} chunks from {path.name}")
    return len(chunks)

## 10. Batch PDF Ingestion

In [10]:
def ingest_data_directory():
    ensure_collection(recreate=False)
    pdf_paths = list(settings.data_dir.glob("*.pdf"))
    if not pdf_paths:
        logger.warning(f"No PDF files found in directory: {settings.data_dir}")
        return 0

    logger.info(f"Found {len(pdf_paths)} PDF files. Proceeding with chunking...")
    chunks = build_chunks(pdf_paths)
    if not chunks:
        return 0

    ids = [str(uuid.uuid5(uuid.NAMESPACE_DNS, c.metadata["chunk_id"]))
        for c in chunks]
    get_vector_store().add_documents(chunks, ids=ids)

    logger.info(f"Ingested {len(chunks)} chunks from {len(pdf_paths)} documents.")
    return len(chunks)

## 11. LLM Dependencies

In [11]:
import torch
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage

## 12. Local Hugging Face LLM

In [12]:
def _build_hf_local():
    logger.info(f"Loading local model: {settings.hf_model}...")
    tokenizer = AutoTokenizer.from_pretrained(settings.hf_model)
    model = AutoModelForCausalLM.from_pretrained(
        settings.hf_model,
        torch_dtype=torch.bfloat16,
        device_map="auto",
    )

    text_gen_pipeline = pipeline(
        task="text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=settings.hf_max_new_tokens,
        do_sample=settings.llm_temperature > 0,
        temperature=settings.llm_temperature,
        return_full_text=False,
    )

    llm = HuggingFacePipeline(pipeline=text_gen_pipeline)
    return ChatHuggingFace(llm=llm)

## 13. Gemini LLM

In [13]:
def _build_gemini():
    return ChatGoogleGenerativeAI(
        model=settings.gemini_model,
        temperature=settings.llm_temperature,
    )

## 14. LLM Provider Selector

In [14]:
def get_llm():
    provider = settings.llm_provider
    if provider == "hf_local":
        return _build_hf_local()
    elif provider == "gemini":
        return _build_gemini()
    else:
        raise ValueError(f"Provider {provider} is not supported.")

## 15. Retrieval and Citations

In [15]:
def retrieve(query: str, k: int = 5):
    store = get_vector_store()
    hits = store.similarity_search_with_score(query=query, k=k)
    return [
        RetrievedChunk(
            text=doc.page_content,
            score=float(score),
            metadata=ChunkMetadata(**doc.metadata),
        )
        for doc, score in hits
    ]

def format_citations(chunks):
    return [
        Citation(
            source_index=i,
            source_marker=f"S{i}",
            filename=c.metadata.filename,
            page=c.metadata.page,
            chunk_id=c.metadata.chunk_id,
        )
        for i, c in enumerate(chunks, start=1)
    ]

## 16. JSON Output Parser

In [16]:
def _parse_json(text: str) -> dict:
    match = re.search(r'\{.*\}', text, re.DOTALL)

    if match:
        json_str = match.group(0)
        try:
            return json.loads(json_str)
        except json.JSONDecodeError as e:
            logger.error(f"Internal JSON parsing error: {e}")
            logger.error(f"Faulty string: {json_str}")
            return {}
    else:
        logger.error(f"No JSON structure found in LLM output: {text}")
        return {}


## 17. Summary Prompt Template

In [17]:
def build_summary_single_prompt(chunks: list) -> str:
    context_text = "\n".join([
        f"---\n[source=S{i+1}] ({chunk.metadata.filename} p.{chunk.metadata.page})\n{chunk.text}"
        for i, chunk in enumerate(chunks)
    ])

    prompt = f"""
You are writing a study-oriented summary grounded strictly in the provided context.

Rules:
- Use only facts explicitly supported by the context. Do not invent details.
- Do not add outside knowledge.
- Focus on concepts, definitions, relationships, and reasoning a learner should retain.
- Keep the tone clear, neutral, and practical for study.
- Write the summary and key points in English.
- If the context is empty or unrelated, return an empty summary and an empty list of key points.

Output STRICTLY valid JSON with this shape and no extra text:
{{
  "summary": "A coherent multi-paragraph study summary.",
  "key_points": ["concise learnable fact", "concise learnable fact"]
}}

Context:
{context_text}

Respond with ONLY the JSON object."""
    return prompt


In [24]:
# query = "Tóm tắt phương pháp Pretraining trong huấn luyện mô hình"
# chunks = retrieve(query, k=2)
# prompt = build_summary_single_prompt(chunks)
# print(prompt)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


You are writing a study-oriented summary grounded strictly in the provided context.

Rules:
- Use only facts explicitly supported by the context. Do not invent details.
- Do not add outside knowledge.
- Focus on concepts, definitions, relationships, and reasoning a learner should retain.
- Keep the tone clear, neutral, and practical for study.
- Write the summary and key points in English.
- If the context is empty or unrelated, return an empty summary and an empty list of key points.

Output STRICTLY valid JSON with this shape and no extra text:
{
  "summary": "A coherent multi-paragraph study summary.",
  "key_points": ["concise learnable fact", "concise learnable fact"]
}

Context:
---
[source=S1] ([Description]-Pretraining-GPT.pdf p.17)
AI VIET NAM (AIO2025) aivietnam.edu.vn
Bảng thuật ngữ
Thuật ngữ Tạm dịch Mô tả
Fine-tuningTinh chỉnh Giai đoạn huấn luyện tiếp theo sau pre-training,
sử dụng tập dữ liệu nhỏ có dán nhãn để cập nhật
trọng số, giúp mô hình thích nghi và giải quyết
tố

## 18. Summary Generation Feature

In [20]:
def summarize_fixed(query: str):
    chunks = retrieve(query, k=settings.summarize_retrieval_k) #default = 12
    prompt = build_summary_single_prompt(chunks)

    response = get_llm().invoke([HumanMessage(content=prompt)])
    payload = _parse_json(response.content)

    return Summary(
        scope="query",
        target=query,
        summary=payload.get("summary", ""),
        key_points=payload.get("key_points", []),
        citations=format_citations(chunks),
    )


In [25]:
# import textwrap

# query = "Tóm tắt phương pháp Pretraining trong huấn luyện mô hình"
# sum_res = summarize_fixed(query)

# summary = textwrap.fill(sum_res.summary, width=80)
# print(summary)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

2026-05-08 01:38:52.573 | INFO     | __main__:_build_hf_local:2 - Loading local model: Qwen/Qwen2.5-1.5B-Instruct...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Fine-tuning is a crucial step after pre-training in large language models like
GPT, where small labeled datasets are used to update weights and improve
specific tasks.  The Generative Pre-trained Transformer (GPT) is a
groundbreaking model introduced by OpenAI in 2018 that uses a decoder-only
transformer architecture without encoders, focusing solely on masked self-
attention for autoregressive predictions. This approach aims to enhance language
understanding through vast unlabeled data.


In [26]:
# print(sum_res.citations)

[Citation(source_index=1, source_marker='S1', filename='[Description]-Pretraining-GPT.pdf', page=17, section=None, chunk_id='d400ca397b9eefcb:17:31'), Citation(source_index=2, source_marker='S2', filename='[Description]-Pretraining-GPT.pdf', page=17, section=None, chunk_id='d400ca397b9eefcb:17:32'), Citation(source_index=3, source_marker='S3', filename='[Description]-Poem-Generation-GPT2.pdf', page=4, section=None, chunk_id='8de594116b66515d:4:4'), Citation(source_index=4, source_marker='S4', filename='[Reading]-LLM-Alignment.pdf', page=15, section=None, chunk_id='c05975b26ff7910a:15:36'), Citation(source_index=5, source_marker='S5', filename='[Description]-LLMs-Fine-tuning.pdf', page=9, section=None, chunk_id='e3c36c65e8104806:9:19'), Citation(source_index=6, source_marker='S6', filename='[Reading]-Reasoning-LLMs.pdf', page=4, section=None, chunk_id='491adecc7920737a:4:6'), Citation(source_index=7, source_marker='S7', filename='[Description]-Poem-Generation-GPT2.pdf', page=16, section

## 19. End-to-End Demo

In [22]:
import urllib.request
import shutil

# 1. Prepare Storage
if settings.storage_dir.exists():
    try:
        client = get_client()
        client.close()
    except Exception:
        logger.warning("Storage is locked. Attempting to clear storage directory for a clean start.")
        shutil.rmtree(settings.storage_dir, ignore_errors=True)
        settings.storage_dir.mkdir(parents=True, exist_ok=True)

settings.data_dir.mkdir(parents=True, exist_ok=True)
logger.info("STEP 1: CHECKING & PREPARING DATA AT /content/data")

if not any(settings.data_dir.iterdir()):
    logger.info("Directory is empty, downloading sample PDF for testing...")
    sample_url = "https://www.w3.org/WAI/ER/tests/xhtml/testfiles/resources/pdf/dummy.pdf"
    urllib.request.urlretrieve(sample_url, settings.data_dir / "sample_dummy.pdf")
else:
    logger.info(f"Found documents in {settings.data_dir}. Skipping sample download.")

# 2. Ingest Documents
logger.info("STEP 2: INGESTING DIRECTORY INTO VECTOR STORE (QDRANT)")
total_chunks = ingest_data_directory()
print(f"Successfully chunked and saved {total_chunks} text segments into Qdrant.\n")

# 3. Run Summary Feature
if total_chunks > 0:
    logger.info("STEP 3: GENERATING DOCUMENT SUMMARY")
    print("\n" + "=" * 55)
    print("FEATURE: DOCUMENT SUMMARY")
    print("=" * 55)

    summary_query = "Tóm tắt phương pháp Pretraining trong huấn luyện mô hình."
    sum_res = summarize_fixed(summary_query)

    print(f"Overall Summary:\n{sum_res.summary}\n")
    print("Key Points:")
    for kp in sum_res.key_points:
        print(f"  * {kp}")

    logger.info("SUMMARY GENERATION COMPLETED!")

2026-05-08 01:36:29.181 | INFO     | __main__:<cell line: 0>:15 - STEP 1: CHECKING & PREPARING DATA AT /content/data
2026-05-08 01:36:29.182 | INFO     | __main__:<cell line: 0>:22 - Found documents in data. Skipping sample download.
2026-05-08 01:36:29.187 | INFO     | __main__:<cell line: 0>:25 - STEP 2: INGESTING DIRECTORY INTO VECTOR STORE (QDRANT)
2026-05-08 01:36:29.410 | INFO     | __main__:ingest_data_directory:8 - Found 7 PDF files. Proceeding with chunking...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

2026-05-08 01:37:18.834 | INFO     | __main__:ingest_data_directory:17 - Ingested 340 chunks from 7 documents.
2026-05-08 01:37:18.836 | INFO     | __main__:<cell line: 0>:31 - STEP 3: GENERATING DOCUMENT SUMMARY


Successfully chunked and saved 340 text segments into Qdrant.


FEATURE: DOCUMENT SUMMARY


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

2026-05-08 01:37:24.895 | INFO     | __main__:_build_hf_local:2 - Loading local model: Qwen/Qwen2.5-1.5B-Instruct...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
2026-05-08 01:37:52.270 | INFO     | __main__:<cell line: 0>:44 - SUMMARY GENERATION COMPLETED!


Overall Summary:
To train a large language model like Llama-3.2-1B-Instruct, we need to follow these steps:

1. Install necessary libraries and set up environment.
2. Load the base model from Hugging Face.
3. Configure LoRA to reduce computational resources.
4. Prepare the dataset including questions and correct answers.
5. Fine-tune the LLM using Gradient Progressive Optimization (GRPO) and LoRA techniques.
6. Save checkpoints during training for monitoring progress and recovery if needed.
7. Run evaluation on test data to assess the model's reading comprehension ability after fine-tuning.

The process involves two stages: pretraining and fine-tuning. In the first stage, the model learns natural language generation rules and foundational knowledge from a large-scale corpus. Afterward, it moves to the second stage where it continues pretraining on a narrower domain such as Vietnamese poetry.

Key Points:
  * Install necessary libraries and set up environment.
  * Load the base model fr